# 06 — Exportação e Preparação para Deploy

**Objetivo:** Gerar os artefatos de consumo do app (features.parquet, personas.json), o relatório HTML (via src/report.py reutilizável) e a checagem de integridade ponta a ponta.

**Inputs:** `models/modelo_final.pkl, models/encoder.pkl, models/scaler.pkl`

**Outputs:** `reports/relatorio_final.html`

---

**Roteiro:**

1. Setup, tema e configurações locais e globais
2. Carregar artefatos gerados nos NBs anteriores
3. features.parquet: base de consumo do app Streamlit
4. Relatório final
5. Checagem final de integridade

### Etapa 1 - Setup, tema e configurações locais e globais

In [1]:
import json

import joblib
import numpy as np
import pandas as pd

from src.config import CONFIG, CAMINHOS
from src.report import RelatorioHTML

DADOS_PRO   = CAMINHOS.dados_processed
MODELS_DIR  = CAMINHOS.modelos
FIG_DIR = CAMINHOS.figures
REPORTS_DIR = CAMINHOS.reports

print("✅ Setup OK\n")
print(f"DADOS_PRO   : {DADOS_PRO}")
print(f"MODELS_DIR  : {MODELS_DIR}")
print(f"FIG_DIR     : {FIG_DIR}")
print(f"REPORTS_DIR : {REPORTS_DIR}")

✅ Setup OK

DADOS_PRO   : C:\Users\jas_t\Repo\Portfolios\customer_segmentation_mall\data\processed
MODELS_DIR  : C:\Users\jas_t\Repo\Portfolios\customer_segmentation_mall\models
FIG_DIR     : C:\Users\jas_t\Repo\Portfolios\customer_segmentation_mall\reports\figures
REPORTS_DIR : C:\Users\jas_t\Repo\Portfolios\customer_segmentation_mall\reports


### Etapa 2 - Carregar artefatos gerados nos NBs anteriores

In [2]:
pipeline_final  = joblib.load(MODELS_DIR / "pipeline_final.joblib")
df_clusterizado = pd.read_parquet(DADOS_PRO / "dados_clusterizados.parquet")

with open(MODELS_DIR / "personas.json", encoding="utf-8") as f:
    PERSONAS_JSON = json.load(f)

print("✅ Artefatos finais carregados")
print(f"   pipeline_final  : {type(pipeline_final.named_steps['kmeans']).__name__}"
      f" (k={pipeline_final.named_steps['kmeans'].n_clusters})")
print(f"   df_clusterizado : {df_clusterizado.shape}  cols={list(df_clusterizado.columns)}")
print(f"   personas.json   : {len(PERSONAS_JSON)} personas")
print(f"\n   Personas:")
for c, info in PERSONAS_JSON.items():
    print(f"     C{c} · {info['nome']}")

✅ Artefatos finais carregados
   pipeline_final  : KMeans (k=6)
   df_clusterizado : (200, 6)  cols=['genero', 'idade', 'renda_anual', 'score_gasto', 'cluster', 'persona']
   personas.json   : 6 personas

   Personas:
     C0 · Maduros Equilibrados
     C1 · Jovens Equilibrados
     C2 · Abastados Conservadores
     C3 · Premium
     C4 · Jovens Impulsivos
     C5 · Econômicos


### Etapa 3 — features.parquet: base de consumo do app Streamlit

In [3]:
features = df_clusterizado.copy()

ORDEM = ["genero", "idade", "renda_anual", "score_gasto", "cluster", "persona"]
features = features[ORDEM]

CAMINHO_FEATURES = DADOS_PRO / "features.parquet"
features.to_parquet(CAMINHO_FEATURES, index=True)

print("═" * 60)
print("FEATURES.PARQUET (base do app)")
print("═" * 60)
print(f"   Salvo: data/processed/features.parquet")
print(f"   Shape: {features.shape}  ·  índice: {features.index.name}")
print(f"   Colunas: {list(features.columns)}")

════════════════════════════════════════════════════════════
FEATURES.PARQUET (base do app)
════════════════════════════════════════════════════════════
   Salvo: data/processed/features.parquet
   Shape: (200, 6)  ·  índice: id_cliente
   Colunas: ['genero', 'idade', 'renda_anual', 'score_gasto', 'cluster', 'persona']


In [4]:
print("\n" + "═" * 75)
print("RESUMO POR PERSONA (preview do que o app mostrará)")
print("═" * 75)
resumo = (features.groupby("persona")
          .agg(n_clientes=("cluster", "size"),
               idade_media=("idade", "mean"),
               renda_media=("renda_anual", "mean"),
               gasto_medio=("score_gasto", "mean"))
          .round(1)
          .sort_values("n_clientes", ascending=False))
print(resumo.to_string())


═══════════════════════════════════════════════════════════════════════════
RESUMO POR PERSONA (preview do que o app mostrará)
═══════════════════════════════════════════════════════════════════════════
                         n_clientes  idade_media  renda_media  gasto_medio
persona                                                                   
Maduros Equilibrados             45         56.3         54.3         49.1
Jovens Equilibrados              39         26.8         57.1         48.1
Premium                          39         32.7         86.5         82.1
Abastados Conservadores          33         41.9         88.9         17.0
Jovens Impulsivos                23         25.0         25.3         77.6
Econômicos                       21         45.5         26.3         19.4


In [5]:
chk = pd.read_parquet(CAMINHO_FEATURES)

print("═" * 75)
print("ROUND-TRIP")
print("═" * 75)
print(f" {'✅' if chk.index.name == 'id_cliente' else '❌'} índice preservado")
print(f" {'✅' if list(chk.columns) == ORDEM else '❌'} ordem de colunas correta")

═══════════════════════════════════════════════════════════════════════════
ROUND-TRIP
═══════════════════════════════════════════════════════════════════════════
 ✅ índice preservado
 ✅ ordem de colunas correta


### Etapa 4 — Relatório final

In [6]:
km = pipeline_final.named_steps["kmeans"]

# ── Resumo por persona (alimenta a tabela) ───────────────────────
resumo = (features.groupby("persona")
          .agg(clientes=("cluster", "size"), idade_media=("idade", "mean"),
               renda_media=("renda_anual", "mean"), gasto_medio=("score_gasto", "mean"))
          .round(1).sort_values("clientes", ascending=False))

# ── Figuras curadas (etapa → legenda) ────────────────────────────
FIGS = [
    (FIG_DIR / "nb01_scatter_grupos.png",   "EDA — Renda × Score de Gasto: estrutura dos grupos"),
    (FIG_DIR / "nb04_selecao_k.png",        "Seleção de k — quatro métricas convergem em k=6"),
    (FIG_DIR / "nb05_perfil_personas.png",  "Perfil das 6 personas (normalizado + radar)"),
    (FIG_DIR / "nb05_arvore_surrogate.png", "Surrogate DecisionTree — regras por persona"),
    (FIG_DIR / "nb05_clusters_pca.png",     "Clusters e centroides no espaço PCA (2D e 3D)"),
]

# ── Montagem encadeada ───────────────────────────────────────────
caminho = (
    RelatorioHTML(
        titulo="Segmentação de Clientes — Mall Customers",
        autor="Jhonnes Toledo",
        subtitulo="Clustering não supervisionado · Pipeline sklearn + KMeans (k=6)",
    )
    .add_secao("1. Visão geral",
        "<div class='card'><p>Segmentação de 200 clientes de shopping em "
        "<strong>6 personas</strong> acionáveis, a partir de idade, renda anual e "
        "score de gasto. Usa um <code>Pipeline</code> sklearn "
        "(<code>StandardScaler → KMeans</code>) como artefato único de deploy, "
        "PCA reservado para visualização e uma DecisionTree surrogate para "
        "interpretabilidade.</p></div>")
    .add_secao("2. Método",
        "<ul>"
        "<li><strong>EDA + estatística:</strong> Shapiro-Wilk (não-normalidade), "
        "Spearman (features independentes), Hopkins = 0.70 (tendência de "
        "clusterização), Mann-Whitney (gênero não diferencia grupos → excluído).</li>"
        "<li><strong>Pré-processamento:</strong> StandardScaler nas 3 numéricas; "
        "<code>genero</code> descartado via <code>remainder='drop'</code>.</li>"
        "<li><strong>Seleção de k:</strong> cotovelo + Silhouette + Davies-Bouldin + "
        "Calinski-Harabasz → <span class='badge'>k = 6</span></li>"
        "<li><strong>Interpretabilidade:</strong> surrogate com 92.5% de fidelidade "
        "em teste.</li>"
        "</ul>")
    .add_secao("3. Métricas finais (k=6)", "")
    .add_metricas({
        "Silhouette ↑":        f"{0.428:.3f}",
        "Davies-Bouldin ↓":    f"{0.825:.3f}",
        "Calinski-Harabasz ↑": f"{135.1:.1f}",
        "Inércia":             f"{km.inertia_:.1f}",
        "Fidelidade surrogate": "92.5%",
    })
    .add_secao("4. As 6 personas", "")
    .add_tabela(resumo, incluir_indice=True)
    .add_secao("5. Visualizações", "")
    .add_figuras(FIGS)
    .add_secao("6. Conclusão",
        "<div class='card'><p>Os clusters são geometricamente válidos e "
        "<strong>explicáveis por regras simples de negócio</strong>. A separação entre "
        "<em>Maduros</em> e <em>Jovens Equilibrados</em> — capturada pela idade — "
        "justificou k=6 sobre o k=5 clássico, confirmada de forma independente pelo "
        "surrogate. O artefato <code>pipeline_final.joblib</code> recebe dados crus e "
        "atribui a persona em uma chamada, pronto para deploy.</p></div>")
    .add_html("<footer>Customer Segmentation · Mall Customers · "
              "Pipeline sklearn + KMeans · github.com/jhastoledo</footer>")
    .salvar(REPORTS_DIR / "relatorio_final.html")
)

tamanho_mb = caminho.stat().st_size / 1e6
print("═" * 60)
print("RELATÓRIO FINAL GERADO")
print("═" * 60)
print(f"   Salvo: reports/relatorio_final.html")
print(f"   Tamanho: {tamanho_mb:.2f} MB · {len(FIGS)} figuras embutidas")
print(f"   Abra no navegador para visualizar.")

════════════════════════════════════════════════════════════
RELATÓRIO FINAL GERADO
════════════════════════════════════════════════════════════
   Salvo: reports/relatorio_final.html
   Tamanho: 1.36 MB · 5 figuras embutidas
   Abra no navegador para visualizar.


### Etapa 5 — Checagem final de integridade

In [7]:
ARTEFATOS_ESPERADOS = {
    "models/pipeline_final.joblib":          MODELS_DIR / "pipeline_final.joblib",
    "models/personas.json":                  MODELS_DIR / "personas.json",
    "models/preprocessor.pkl":               MODELS_DIR / "preprocessor.pkl",
    "models/pca_visualizacao.pkl":           MODELS_DIR / "pca_visualizacao.pkl",
    "data/processed/features.parquet":       DADOS_PRO / "features.parquet",
    "data/processed/dados_clusterizados.parquet": DADOS_PRO / "dados_clusterizados.parquet",
    "reports/relatorio_final.html":          REPORTS_DIR / "relatorio_final.html",
}

print("═" * 70)
print("CHECAGEM DE INTEGRIDADE — ARTEFATOS DE DEPLOY")
print("═" * 70)
todos_ok = True
for nome, caminho in ARTEFATOS_ESPERADOS.items():
    existe = caminho.exists()
    todos_ok &= existe
    tam = f"{caminho.stat().st_size/1e3:.0f} KB" if existe else "—"
    print(f"   {'✅' if existe else '❌'} {nome:<46} {tam:>10}")

# ── Teste funcional: pipeline atribui persona a um cliente novo ──
print("\n" + "═" * 70)
print("TESTE FUNCIONAL — inferência ponta a ponta")
print("═" * 70)
cliente_novo = pd.DataFrame([{
    "genero": "Female", "idade": 30, "renda_anual": 90, "score_gasto": 85
}])
cluster_previsto = int(pipeline_final.predict(cliente_novo)[0])
persona_prevista = PERSONAS_JSON[str(cluster_previsto)]["nome"]
print(f"   Cliente: 30 anos, renda 90k, gasto 85")
print(f"   → Cluster {cluster_previsto} · {persona_prevista}")
print(f"   (esperado: Premium — renda alta + gasto alto)")

print("\n" + "═" * 70)
print(f"   {'✅ PROJETO PRONTO PARA DEPLOY' if todos_ok else '❌ ARTEFATOS FALTANDO'}")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
CHECAGEM DE INTEGRIDADE — ARTEFATOS DE DEPLOY
══════════════════════════════════════════════════════════════════════
   ✅ models/pipeline_final.joblib                         4 KB
   ✅ models/personas.json                                 1 KB
   ✅ models/preprocessor.pkl                              2 KB
   ✅ models/pca_visualizacao.pkl                          1 KB
   ✅ data/processed/features.parquet                      7 KB
   ✅ data/processed/dados_clusterizados.parquet           7 KB
   ✅ reports/relatorio_final.html                      1358 KB

══════════════════════════════════════════════════════════════════════
TESTE FUNCIONAL — inferência ponta a ponta
══════════════════════════════════════════════════════════════════════
   Cliente: 30 anos, renda 90k, gasto 85
   → Cluster 3 · Premium
   (esperado: Premium — renda alta + gasto alto)

═════════════════════════════════════════════════════════════════════